In [3]:
#importing pandas
import numpy as np
import pandas as pd
pd.__version__

'2.2.2'

In [7]:
#loading data from drive
from google.colab import drive
#mounting drive directory root
drive.mount ('/content/drive')

# loading data
# for csv file pd.read_csv

# loading csv  from workbook

df = pd.read_csv('/content/drive/My Drive/Colab Notebooks/byteverse26/rawDataset/KaggleDiseaseAndSymptoms.csv')

#calling top 5 records
df.sample(5)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
3422,Hepatitis C,fatigue,yellowish_skin,nausea,loss_of_appetite,yellowing_of_eyes,family_history,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2128,Chronic cholestasis,itching,vomiting,yellowish_skin,nausea,loss_of_appetite,abdominal_pain,yellowing_of_eyes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3645,Psoriasis,skin_rash,joint_pain,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3461,hepatitis A,joint_pain,vomiting,yellowish_skin,dark_urine,nausea,loss_of_appetite,abdominal_pain,diarrhoea,mild_fever,yellowing_of_eyes,muscle_pain,NaN,NaN,NaN,NaN,NaN,NaN
1427,Malaria,chills,vomiting,high_fever,sweating,headache,diarrhoea,muscle_pain,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


removing spaces
 and converting into lower case

In [8]:
df.columns = df.columns.str.strip().str.lower()
df.columns.unique()

Index(['disease', 'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4',
       'symptom_5', 'symptom_6', 'symptom_7', 'symptom_8', 'symptom_9',
       'symptom_10', 'symptom_11', 'symptom_12', 'symptom_13', 'symptom_14',
       'symptom_15', 'symptom_16', 'symptom_17'],
      dtype='object')

In [10]:
#detect disease column
possible_names = ["disease"]

disease_col = None
for col in df.columns:
    if col in possible_names:
        disease_col = col
        break


In [11]:
#clean text
def clean_text(x):
    if isinstance(x, str):
        return (
            x.strip()
            .replace("_", " ")
            .replace(",", " ")
            .capitalize()
        )
    return x

df = df.map(clean_text)

#replace null
df.replace(["", "none", "nan"], pd.NA, inplace=True)
df


,disease,symptom_1,symptom_2,symptom_3,symptom_4,symptom_5,symptom_6,symptom_7,symptom_8,symptom_9,symptom_10,symptom_11,symptom_12,symptom_13,symptom_14,symptom_15,symptom_16,symptom_17
0,Influenza (flu),Fever,Chills,Body ache,Fatigue,Dry cough,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Common cold,Runny nose,Sneezing,Sore throat,Mild cough,Congestion,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Viral fever,Fever,Headache,Fatigue,Body pain,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Food poisoning,Vomiting,Diarrhea,Abdominal pain,Nausea,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Gastritis,Stomach pain,Bloating,Nausea,Indigestion,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4954,(vertigo) paroymsal positional vertigo,Vomiting,Headache,Nausea,Spinning movements,Loss of balance,Unsteadiness,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4955,Acne,Skin rash,Pus filled pimples,Blackheads,Scurring,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4956,Urinary tract infection,Burning micturition,Bladder discomfort,Foul smell of urine,Continuous feel of urine,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4957,Psoriasis,Skin rash,Joint pain,Skin peeling,Silver like dusting,Small dents in nails,Inflammatory nails,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# mould dataframe to new one now
df_long = df.melt(
    id_vars=[disease_col],
    var_name="symptom_slot",
    value_name="symptom"
)

In [13]:
# dropping empty tuples if any
df_long = df_long.dropna(subset=["symptom"])


In [14]:
# new table disease only
disease_table = (
    df_long[[disease_col]]
    .drop_duplicates()
    .reset_index(drop=True)
)

disease_table["disease_id"] = disease_table.index + 1

In [15]:
#symptoms
symptom_table = (
    df_long[["symptom"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

symptom_table["symptom_id"] = symptom_table.index + 1

In [16]:
# relation b/w two
mapping = df_long.merge(disease_table, on=disease_col)
mapping = mapping.merge(symptom_table, on="symptom")

mapping_table = (
    mapping[["disease_id", "symptom_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [17]:
disease_table

,disease,disease_id
0,Influenza (flu),1
1,Common cold,2
2,Viral fever,3
3,Food poisoning,4
4,Gastritis,5
...,...,...
73,(vertigo) paroymsal positional vertigo,74
74,Acne,75
75,Urinary tract infection,76
76,Psoriasis,77


In [18]:
symptom_table

,symptom,symptom_id
0,Fever,1
1,Runny nose,2
2,Vomiting,3
3,Stomach pain,4
4,Heartburn,5
...,...,...
172,Coma,173
173,Sinus pressure,174
174,Stomach bleeding,175
175,Blood in sputum,176


In [19]:
mapping_table

,disease_id,symptom_id
0,1,1
1,2,2
2,3,1
3,4,3
4,5,4
...,...,...
465,64,22
466,2,22
467,64,176
468,2,177


In [20]:
# saving the data into seperate tables
disease_table.to_csv("/content/drive/My Drive/Colab Notebooks/byteverse26/extractedTables/disease_table.csv", index=False)
symptom_table.to_csv("/content/drive/My Drive/Colab Notebooks/byteverse26/extractedTables/symptom_table.csv", index=False)
mapping_table.to_csv("/content/drive/My Drive/Colab Notebooks/byteverse26/extractedTables/disease_symptom_mapping.csv", index=False)

just mapping will not work not adding weights to the symptoms as well again.
more frequency in mapping tabel for symptoms for multiple disease leads to less weight

In [21]:
symptom_freq = mapping_table['symptom_id'].value_counts().reset_index()
symptom_freq.columns = ['symptom_id', 'count']
symptom_freq

,symptom_id,count
0,17,28
1,3,19
2,6,16
3,19,16
4,18,12
...,...,...
172,173,1
173,174,1
174,175,1
175,176,1


In [22]:
def assign_weight(count):
    if count > 30:
        return 1   # very common → weak
    elif count > 15:
        return 2   # medium
    elif count > 5:
        return 3   # useful
    else:
        return 5   # rare → strong

symptom_freq['weight'] = symptom_freq['count'].apply(assign_weight)

In [23]:
symptom_freq

,symptom_id,count,weight
0,17,28,2
1,3,19,2
2,6,16,2
3,19,16,2
4,18,12,3
...,...,...,...
172,173,1,5
173,174,1,5
174,175,1,5
175,176,1,5


In [24]:
mapping_table

,disease_id,symptom_id
0,1,1
1,2,2
2,3,1
3,4,3
4,5,4
...,...,...
465,64,22
466,2,22
467,64,176
468,2,177


In [25]:
mapping_weighted = mapping_table.merge(
    symptom_freq[['symptom_id', 'weight']],
    on='symptom_id',
    how='left'
)
mapping_weighted

,disease_id,symptom_id,weight
0,1,1,3
1,2,2,5
2,3,1,3
3,4,3,2
4,5,4,5
...,...,...,...
465,64,22,3
466,2,22,3
467,64,176,5
468,2,177,5


In [26]:
mapping_weighted.to_csv("/content/drive/My Drive/Colab Notebooks/byteverse26/extractedTables/disease_symptom_mapping_weighted.csv", index=False)